# Many repositories

> Every repository a workspace can see, listed in one stable order, with one operation across all of them.

A workspace is a list of open folders. `discover` turns those folders into repository roots: a folder that is itself a checkout, every direct child that is one, and whatever a `repos.txt` beside it names. `RepoSet` layers on what a person added or hid, and runs fetch, pull, push or status across the whole list.

Git is a subprocess, so a fan-out across repositories is IO and not compute. Every operation here runs on a thread pool, and every row carries its own failure rather than aborting the set.

In [ ]:
#| default_exp repos

In [ ]:
#| export
from __future__ import annotations

import hashlib, json, os, re, shutil, subprocess
from collections import Counter
from dataclasses import dataclass

from fastcore.all import Path, first, patch
from fastcore.parallel import parallel
from fastcore.xdg import xdg_config_home

from gheasy.repo import GitError, GitRepo, repo_root, url_name
from gheasy.repo import clone as git_clone

In [ ]:
#| hide
import atexit, shutil as _shutil, tempfile
from fastcore.test import *

_tmp = []
atexit.register(lambda: [_shutil.rmtree(d, ignore_errors=True) for d in _tmp])

def sh(cwd, *args):
    "Run Git directly when preparing a test fixture."
    return subprocess.run(['git', *args], cwd=str(cwd), check=True, text=True,
                          capture_output=True).stdout.strip()

def mkrepo(where, name):
    "A repository with one commit, at `where/name`."
    d = Path(where)/name
    d.mkdir(parents=True)
    sh(d, 'init', '-b', 'main')
    sh(d, 'config', 'user.email', 'tests@example.com')
    sh(d, 'config', 'user.name', 'Repo tests')
    sh(d, 'config', 'commit.gpgsign', 'false')
    (d/'readme.md').write_text(f'{name}\n')
    sh(d, 'add', '-A')
    sh(d, 'commit', '-m', 'initial')
    return d

def mkfolder(*names):
    "A throwaway folder holding one repository per name."
    d = Path(tempfile.mkdtemp()).resolve()
    _tmp.append(str(d))
    for n in names: mkrepo(d, n)
    return d

def state():
    "A throwaway store, so no test reads or writes the real `STATE_DIR`."
    d = Path(tempfile.mkdtemp()); _tmp.append(str(d))
    return d/'state.json'

## What a folder names

A folder can list repositories that are not under it. `repos.txt` beside it holds one per line, and a line is either a path relative to the folder, an absolute path, or the URL a clone came from. A URL line resolves to the folder that clone would have landed in, so a list can be shared between people who have not all checked everything out.

`clone_url` is the other half of that shorthand: `owner/name` is what a list is written with, anything else is already a URL.

In [ ]:
#| export
MEMBERS = 'repos.txt'                            #: one repository per line, `#` starts a comment
OPS = ('fetch', 'pull', 'push', 'status')
WORKERS = 8
STATE_DIR = xdg_config_home()/'gheasy'/'repos'   #: where a `RepoSet` remembers what was added or hidden

def _members_file(root):
    "The entries of the `repos.txt` beside an open folder."
    try: text = (Path(root)/MEMBERS).read_text(encoding='utf-8', errors='replace')
    except OSError: return []
    lines = (line.split('#', 1)[0].strip() for line in text.splitlines())
    return [line for line in lines if line]

def _member_root(entry, folder):
    "The local worktree a `repos.txt` line names, or None if it is not checked out here."
    entry = str(entry).strip()
    if not entry: return None
    if entry.startswith(('http://', 'https://', 'git@', 'ssh://', 'git://')):
        candidate = Path(folder)/url_name(entry)
    else:
        candidate = Path(entry).expanduser()
        if not candidate.is_absolute(): candidate = Path(folder)/candidate
        if not candidate.is_dir() and '/' in entry:
            candidate = Path(folder)/entry.rstrip('/').rpartition('/')[2]
    return repo_root(candidate) if candidate.is_dir() else None

def clone_url(spec):
    "`owner/name` is the shorthand a list is written with; anything else is already a URL."
    spec = str(spec or '').strip()
    return f'https://github.com/{spec}.git' if re.fullmatch(r'[\w.-]+/[\w.-]+', spec) else spec

In [ ]:
folder = mkfolder('alpha')
mkrepo(folder/'nested', 'deep')
(folder/MEMBERS).write_text('alpha  # the note after a repository\n# a whole line\n\nnested/deep\n')
test_eq(_members_file(folder), ['alpha', 'nested/deep'])
test_eq(_members_file(folder/'nowhere'), [])            # no list at all reads as an empty one

test_eq(_member_root('nested/deep', folder), folder/'nested'/'deep')
test_eq(_member_root('https://github.com/o/alpha.git', folder), folder/'alpha')
test_eq(_member_root('https://github.com/o/absent.git', folder), None)   # not checked out here

test_eq(clone_url('answerdotai/fastcore'), 'https://github.com/answerdotai/fastcore.git')
test_eq(clone_url('git@github.com:answerdotai/fastcore.git'), 'git@github.com:answerdotai/fastcore.git')
clone_url('https://github.com/answerdotai/fastcore.git')

## The order they come back in

The order is fixed so that a list a person has learnt does not move under them: the open folder itself, then its direct children by name, then what its `repos.txt` names, then the folders another workspace added, with anything hidden dropped at the end. A repository reached twice is kept once, at its first position.

In [ ]:
#| export
def discover(roots, extra=(), hidden=()):
    "Repository roots for a workspace of folders, in a stable, explainable order."
    out, seen = [], set()
    def take(path):
        if path is None or str(path) in seen: return
        seen.add(str(path))
        out.append(Path(path))
    for folder in roots:
        folder = Path(folder).expanduser().resolve()
        take(repo_root(folder))
        try:
            children = sorted((c for c in folder.iterdir() if c.is_dir() and not c.name.startswith('.')),
                key=lambda c: c.name.lower())
        except OSError:
            children = []
        for child in children:
            if (child/'.git').exists(): take(child)
        for entry in _members_file(folder): take(_member_root(entry, folder))
    for path in extra: take(repo_root(Path(path).expanduser()))
    away = {str(Path(h)) for h in hidden}
    return [p for p in out if str(p) not in away]

In [ ]:
folder = mkfolder('beta', 'alpha')
mkrepo(folder/'nested', 'deep')                         # two levels down, so not a direct child
test_eq([p.name for p in discover([folder])], ['alpha', 'beta'])
(folder/MEMBERS).write_text('nested/deep\nalpha\n')
test_eq([p.name for p in discover([folder])], ['alpha', 'beta', 'deep'])   # `alpha` is already in

solo = mkfolder()
top = mkrepo(solo, 'top')
test_eq(discover([top]), [top])                         # an open folder that is itself a checkout
test_eq([p.name for p in discover([folder], extra=[top])], ['alpha', 'beta', 'deep', 'top'])
test_eq([p.name for p in discover([folder], hidden=[folder/'beta'])], ['alpha', 'deep'])
test_eq(discover([solo/'gone']), [])                    # a folder that is not there any more
discover([folder, top])

## The set a workspace lists

`RepoSet` is the open folders plus a store. The store remembers only the two things discovery cannot work out: repositories a person added from outside the folders, and discovered ones they hid. It is named after a hash of the roots, so two windows on the same folders share one list, and a different set of folders gets its own.

Nothing is ever deleted from disk. `remove` hides a discovered repository and drops an added one.

In [ ]:
#| export
@dataclass
class RepoSet:
    "The repositories a workspace lists, and the operations that mean something across all of them."
    roots: list
    store: Path = None                       #: defaults to a file under `STATE_DIR`, named after the roots
    def __post_init__(self):
        self.roots = [Path(r).expanduser().resolve() for r in self.roots]
        key = hashlib.sha256('\0'.join(map(str, self.roots)).encode()).hexdigest()[:16]
        self.store = Path(self.store) if self.store else STATE_DIR/f'{key}.json'

@patch
def _state(self:RepoSet):
    try: return json.loads(self.store.read_text(encoding='utf-8')) or {}
    except (OSError, ValueError): return {}

@patch
def _save(self:RepoSet, state):
    self.store.parent.mkdir(parents=True, exist_ok=True)
    self.store.write_text(json.dumps(state, indent=1), encoding='utf-8')

@patch
def paths(self:RepoSet):
    "Every repository this workspace lists."
    state = self._state()
    return discover(self.roots, state.get('extra') or (), state.get('hidden') or ())

@patch
def require(self:RepoSet, path):
    "The repository a caller named, refused unless this workspace already lists it."
    want = str(Path(str(path)).expanduser().resolve())
    found = first(p for p in self.paths() if str(p) == want)
    if found is None: raise GitError(f'{path} is not a repository in this workspace')
    return found

@patch
def add(self:RepoSet, path):
    "List a repository this workspace would not have found, and un-hide one it removed."
    root = repo_root(Path(str(path)).expanduser())
    if root is None: raise GitError(f'{path} is not inside a Git repository')
    state = self._state()
    state['hidden'] = [h for h in (state.get('hidden') or []) if h != str(root)]
    extra = state.get('extra') or []
    if str(root) not in extra and root not in discover(self.roots): extra += [str(root)]
    state['extra'] = extra
    self._save(state)
    return root

@patch
def remove(self:RepoSet, path):
    "Drop a repository from the list. A discovered one is hidden; nothing is deleted."
    root = self.require(path)
    state = self._state()
    state['extra'] = [e for e in (state.get('extra') or []) if e != str(root)]
    if root in discover(self.roots, state['extra']):
        state['hidden'] = sorted({*(state.get('hidden') or []), str(root)})
    self._save(state)
    return root

In [ ]:
folder, away = mkfolder('alpha', 'beta'), mkfolder('solo')
ws = RepoSet([folder], state())
test_eq([p.name for p in ws.paths()], ['alpha', 'beta'])
test_eq(ws.require(folder/'alpha'), folder/'alpha')
test_fail(lambda: ws.require(away/'solo'), contains='not a repository in this workspace')

ws.add(away/'solo')                                     # from outside the open folders
test_eq([p.name for p in ws.paths()], ['alpha', 'beta', 'solo'])
ws.remove(away/'solo')
test_eq([p.name for p in ws.paths()], ['alpha', 'beta'])

ws.remove(folder/'beta')                                # a discovered one is hidden, not deleted
test_eq([p.name for p in ws.paths()], ['alpha'])
test_eq((folder/'beta'/'.git').is_dir(), True)
ws.add(folder/'beta')                                   # ...and adding it back un-hides it
test_eq([p.name for p in ws.paths()], ['alpha', 'beta'])
test_eq(ws._state(), {'hidden': [], 'extra': []})       # nothing discovery can work out itself
test_fail(lambda: ws.add(away), contains='not inside a Git repository')

In [ ]:
#| hide
ws = RepoSet([folder], state())
test_eq(ws._state(), {})                                # a store that is not there reads as empty
ws.store.write_text('not json')
test_eq(ws._state(), {})                                # ...and so does one that is unreadable
test_eq(RepoSet([folder]).store.parent, STATE_DIR)      # the default, which no test writes to
test_ne(RepoSet([folder]).store, RepoSet([folder, away]).store)   # keyed by the roots
test_eq(RepoSet([folder]).store, RepoSet([str(folder)]).store)

## One row per repository

`overview` is the whole workspace as one table. `expected_branch` is the branch most of the repositories are on unless a caller names one, and `off_branch` marks the rest: a set of repositories worked on together drifts one repository at a time, and that is what a person needs to see first.

A repository that cannot be read becomes an error row rather than an exception, so one broken checkout does not blank the table.

In [ ]:
#| export
def _fan(items, work):
    "Run `work` over every item at once. Git is a subprocess, so this is IO, not compute."
    items = list(items)
    return list(parallel(work, items, n_workers=min(WORKERS, len(items)), threadpool=True))

@patch
def overview(self:RepoSet, expect=''):
    "Every repository as one row, plus what the workspace looks like as a whole."
    def brief(path):
        try:
            return GitRepo(path).brief()
        except (GitError, OSError) as e:
            return {'root': str(path), 'name': Path(path).name, 'error': str(e),
                'branch': '', 'clean': True, 'changed': 0, 'ahead': 0, 'behind': 0,
                'unreleased': None}
    rows = _fan(self.paths(), brief)
    named = [r['branch'] for r in rows if r.get('branch') and not r.get('error')]
    expected = str(expect or '').strip() or (Counter(named).most_common(1)[0][0] if named else '')
    for row in rows:
        row['off_branch'] = bool(expected) and not row.get('error') and row['branch'] != expected
    return {
        'repos': rows, 'expected_branch': expected, 'roots': [str(r) for r in self.roots],
        'summary': {
            'repos': len(rows),
            'dirty': sum(1 for r in rows if not r.get('clean')),
            'ahead': sum(1 for r in rows if r.get('ahead')),
            'behind': sum(1 for r in rows if r.get('behind')),
            'off_branch': sum(1 for r in rows if r.get('off_branch')),
            'unreleased': sum(1 for r in rows if r.get('unreleased')),
            'errors': sum(1 for r in rows if r.get('error')),
        },
    }

In [ ]:
folder = mkfolder('one', 'three', 'two')
(folder/'two'/'readme.md').write_text('edited\n')
sh(folder/'two', 'checkout', '-q', '-b', 'topic')
view = RepoSet([folder], state()).overview()
test_eq([r['name'] for r in view['repos']], ['one', 'three', 'two'])
test_eq([r['clean'] for r in view['repos']], [True, True, False])
test_eq(view['expected_branch'], 'main')                # what most of them are on
test_eq([r['off_branch'] for r in view['repos']], [False, False, True])
view['summary']

In [ ]:
#| hide
test_eq(RepoSet([folder], state()).overview(expect='topic')['summary']['off_branch'], 2)
empty = RepoSet([mkfolder()], state()).overview()
test_eq(empty['expected_branch'], '')                   # nothing to take a majority of
test_eq(empty['summary']['repos'], 0)
test_eq(_fan([], lambda x: x), [])                      # no repositories is not zero workers

## One operation across all of them

`fetch`, `pull` and `push` are the three that mean the same thing in every repository. Each one decides per repository whether there is anything to do, and says so in `skipped` rather than running a command that would fail: a branch with no upstream is not an error, it is a branch nobody has published yet.

`status` runs nothing. It is the operation that just lists, and it is here so a caller has one code path for all four.

In [ ]:
#| export
@patch
def run(self:RepoSet, op, paths=()):
    "One operation across the named repositories, or across all of them."
    if op not in OPS: raise GitError(f'unknown workspace operation: {op}')
    targets = [self.require(p) for p in paths] if paths else self.paths()
    def act(path):
        row = {'root': str(path), 'name': Path(path).name, 'ok': True, 'skipped': '', 'output': ''}
        try:
            repo = GitRepo(path)
            if op == 'status': return row
            if op == 'fetch': return row | {'output': repo.fetch() or 'up to date'}
            if op == 'pull':
                repo.fetch()
                state = repo.brief(fresh=True)
                if not state['upstream']: return row | {'skipped': 'no upstream'}
                if not state['behind']: return row | {'skipped': 'already current'}
                return row | {'output': repo.pull()['summary']}
            state = repo.brief(fresh=True)
            if not state['upstream']: return row | {'skipped': 'unpublished branch'}
            if not state['ahead']: return row | {'skipped': 'nothing to push'}
            return row | {'output': repo.push() or 'pushed'}
        except (GitError, OSError) as e:
            return row | {'ok': False, 'error': str(e)}
    return _fan(targets, act)

In [ ]:
folder = mkfolder('alpha', 'beta')
ws = RepoSet([folder], state())
test_fail(lambda: ws.run('rebase'), contains='unknown workspace operation')
test_eq([r['name'] for r in ws.run('status')], ['alpha', 'beta'])
test_eq([r['skipped'] for r in ws.run('push')], ['unpublished branch'] * 2)
test_eq([r['skipped'] for r in ws.run('pull')], ['no upstream'] * 2)
test_eq([r['output'] for r in ws.run('fetch')], ['up to date'] * 2)

bare = Path(tempfile.mkdtemp()); _tmp.append(str(bare))
sh(bare, 'init', '-q', '--bare', 'origin.git')
sh(folder/'alpha', 'remote', 'add', 'origin', str(bare/'origin.git'))
sh(folder/'alpha', 'push', '-q', '-u', 'origin', 'main')
ws.run('pull', [folder/'alpha'])                        # ...and a published one has an upstream to ask about

In [ ]:
#| hide
test_eq([r['skipped'] for r in ws.run('push', [folder/'alpha'])], ['nothing to push'])
test_fail(lambda: ws.run('status', [bare]), contains='not a repository in this workspace')
(folder/'beta'/'.git'/'HEAD').write_text('garbage\n')   # a checkout that cannot be read
row = first(r for r in ws.run('push') if r['name'] == 'beta')
test_eq(row['ok'], False)
assert row['error']

## Cloning into the workspace

A clone has to land somewhere the workspace already watches, or it would not appear in the list it was asked for. `clone` refuses a parent outside the open folders, and reports each URL's outcome in its own row so one bad name does not lose the others.

In [ ]:
#| export
@patch
def clone(self:RepoSet, urls, into=None):
    "Clone repositories into an open folder, in parallel, and say where each landed."
    parent = Path(str(into)).expanduser().resolve() if into else (self.roots[0] if self.roots else None)
    if parent is None: raise GitError('open a folder to clone into')
    if not any(parent == r or parent.is_relative_to(r) for r in self.roots):
        raise GitError(f'{parent} is outside the open folders')
    specs = [s for s in (str(u).strip() for u in (urls or ())) if s]
    if not specs: raise GitError('give at least one repository to clone')
    def act(spec):
        row = {'spec': spec, 'ok': True, 'skipped': '', 'output': ''}
        try:
            return row | {'root': str(git_clone(clone_url(spec), parent)), 'output': 'cloned'}
        except (GitError, OSError) as e:
            return row | {'ok': False, 'error': str(e)}
    return _fan(specs, act)

In [ ]:
folder, source = mkfolder('alpha'), mkfolder('gamma')
ws = RepoSet([folder], state())
rows = ws.clone([str(source/'gamma')])                   # a local path is a URL git understands
test_eq([r['output'] for r in rows], ['cloned'])
test_eq([p.name for p in ws.paths()], ['alpha', 'gamma'])

test_fail(lambda: ws.clone([]), contains='at least one repository')
test_fail(lambda: ws.clone(['o/r'], into=source), contains='outside the open folders')
test_fail(lambda: RepoSet([], state()).clone(['o/r']), contains='open a folder to clone into')
ws.clone([str(folder/'not-here')])                       # one bad URL is one bad row

## The repository a view is about

A view showing one repository has three things to ask, in order: the one it was told, the open folder if that folder is itself a checkout, and otherwise the repository the open file is in.

The middle step is why a folder of checkouts works. A directory of clones, opened to work across them, is not a repository; answering with it made every Git view report that the folder was not one while every file open inside it was. Falling through to `cwd` is what makes a view follow the file from repository to repository.

In [ ]:
#| export
def repo_for(roots, target='', chosen=None, cwd=None, store=None):
    "The repository a view is about: one this workspace lists, the chosen folder's, or the open file's."
    roots = [Path(r).expanduser().resolve() for r in roots]
    if str(target or '').strip(): return RepoSet(roots, store).require(target)
    if chosen is not None:
        chosen = Path(chosen).expanduser().resolve()
        if chosen in roots and (root := repo_root(chosen)) is not None: return root
    if cwd is None: raise GitError('open a folder first')
    return repo_root(cwd) or Path(cwd)

In [ ]:
folder, st = mkfolder('alpha', 'beta'), state()
solo = mkfolder(); top = mkrepo(solo, 'top')
test_eq(repo_for([folder], target=folder/'alpha', store=st), folder/'alpha')
test_fail(lambda: repo_for([folder], target=solo, store=st), contains='not a repository in this workspace')

test_eq(repo_for([top], chosen=top), top)               # an open folder that is itself a checkout
test_fail(lambda: repo_for([folder], chosen=folder), contains='open a folder first')
test_eq(repo_for([folder], chosen=folder, cwd=folder/'beta'/'readme.md'), folder/'beta')
repo_for([folder], cwd=solo)                            # outside every repository: the folder itself

## Finding one anywhere

`find_repos` is for picking a repository out of a directory of checkouts nobody has listed yet. It prefers `ripgrep`, which walks a large tree far faster than Python can, and falls back to `os.walk` where ripgrep is not installed. Both count `depth` the same way: how many folders below `base` the repository itself is.

In [ ]:
#| export
def find_repos(base, query='', limit=60, depth=6):
    "Repositories anywhere under `base`, for picking one out of a directory of checkouts."
    base = Path(base or '').expanduser()
    if not base.is_dir(): return []
    q, out, seen = str(query or '').strip().lower(), [], set()
    def take(repo):
        key = str(repo)
        rel = os.path.relpath(key, base).lower()   # relative to `base`: an ancestor's name is not a match
        if key in seen or (q and q not in repo.name.lower() and q not in rel): return
        seen.add(key)
        out.append(repo)
    exe = shutil.which('rg')
    if exe:
        # `--max-depth` counts files, and a repository's marker is `<repo>/.git/HEAD`, two below it.
        cmd = [exe, '--files', '--hidden', '--no-ignore', '--no-messages',
            '--max-depth', str(int(depth) + 2), '-g', '**/.git/HEAD', '-g', '**/.git',
            *[g for d in ('.git/objects', '.git/logs', 'node_modules', '.venv', 'venv',
                '.cache', 'Library', '.Trash', 'site-packages', '__pycache__')
                for g in ('-g', f'!**/{d}')],
            str(base)]
        try:
            p = subprocess.run(cmd, capture_output=True, text=True, timeout=20)
            for line in (p.stdout or '').splitlines():
                hit = Path(line)
                repo = hit.parent.parent if hit.name == 'HEAD' else hit.parent
                if repo.is_dir(): take(repo)
        except (OSError, subprocess.SubprocessError):
            exe = None
    if not exe:
        for cur, dirs, _ in os.walk(base):
            here = Path(cur)
            if len(here.relative_to(base).parts) > depth:
                dirs[:] = []
                continue
            if (here/'.git'/'HEAD').is_file() or (here/'.git').is_file():
                take(here)
                dirs[:] = []
                continue
            dirs[:] = [d for d in dirs if not d.startswith('.') and d not in {'node_modules', '__pycache__'}]
    out.sort(key=lambda p: (p.name.lower(), str(p)))
    return out[:max(1, int(limit))]

In [ ]:
base = mkfolder('alpha', 'beta')
mkrepo(base/'nested', 'deep')
test_eq([p.name for p in find_repos(base)], ['alpha', 'beta', 'deep'])
test_eq([p.name for p in find_repos(base, 'de')], ['deep'])         # matched on name or path
test_eq([p.name for p in find_repos(base, depth=1)], ['alpha', 'beta'])
test_eq(len(find_repos(base, limit=1)), 1)
test_eq(find_repos(base/'nowhere'), [])
find_repos(base, depth=2)

In [ ]:
#| hide
saved = shutil.which
try:                                                    # the walk, where ripgrep is not installed
    shutil.which = lambda name: None
    test_eq([p.name for p in find_repos(base)], ['alpha', 'beta', 'deep'])
    test_eq([p.name for p in find_repos(base, depth=1)], ['alpha', 'beta'])
    test_eq([p.name for p in find_repos(base, 'de')], ['deep'])
    test_eq(find_repos(base/'nowhere'), [])
finally: shutil.which = saved

In [ ]:
#| hide
wt = mkfolder('one')                                    # a linked worktree carries `.git` as a file
(wt/'two').mkdir(); (wt/'two'/'.git').write_text('gitdir: /elsewhere/.git/worktrees/two\n')
found = lambda: [p.name for p in find_repos(wt)]
test_eq(found(), ['one', 'two'])
saved = shutil.which
try:                                                    # and both branches have to agree it is one
    shutil.which = lambda name: None
    test_eq(found(), ['one', 'two'])
finally: shutil.which = saved

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()